###Import libraries

In [1]:
!pip install transformers datasets scikit-learn torch -q

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

###Load and prepare dataset

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "ButterChicken98/plantvillage-image-text-pairs",
    split="train"
)

df = dataset.to_pandas()
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/365 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20638 [00:00<?, ? examples/s]

,image,caption,captions
0,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
1,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato Late blight,[A tomato leaf showing dark brown lesions and ...
2,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato healthy,[A vibrant green and healthy tomato leaf with ...
3,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Tomato mosaic virus,[A tomato leaf with mosaic-like patterns of li...
4,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...,Pepper bell healthy,"[A fresh green bell pepper leaf with a smooth,..."


In [4]:
df.rename(columns={'caption': 'class_name'}, inplace=True)

In [5]:
df['text'] = df['captions'].apply(lambda x: x[0])

In [6]:
df = df[['text', 'class_name']]
df.head()

,text,class_name
0,A vibrant green and healthy tomato leaf with s...,Tomato healthy
1,A tomato leaf showing dark brown lesions and w...,Tomato Late blight
2,A vibrant green and healthy tomato leaf with s...,Tomato healthy
3,A tomato leaf with mosaic-like patterns of lig...,Tomato mosaic virus
4,"A fresh green bell pepper leaf with a smooth, ...",Pepper bell healthy


### Encode labels and split data

In [7]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['class_name'])

num_labels = len(label_encoder.classes_)
print("Number of classes:", num_labels)

Number of classes: 15


In [8]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

###Tokenize

In [9]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

In [11]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'labels': val_labels.tolist()
})

###Define model and training arguments

In [12]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",      # NEW correct keyword
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Train model

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.006247,0.002095
2,0.001315,0.000591
3,0.000596,0.000288
4,0.000351,0.000184
5,0.000272,0.000153


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=5160, training_loss=0.03410144058135591, metrics={'train_runtime': 12410.9883, 'train_samples_per_second': 6.651, 'train_steps_per_second': 0.416, 'total_flos': 555431039631000.0, 'train_loss': 0.03410144058135591, 'epoch': 5.0})

###Evaluate model

In [15]:
predictions = trainer.predict(val_dataset)

preds = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(val_labels, preds))
print(classification_report(val_labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       199
           1       1.00      1.00      1.00       296
           2       1.00      1.00      1.00       200
           3       1.00      1.00      1.00       200
           4       1.00      1.00      1.00        30
           5       1.00      1.00      1.00       426
           6       1.00      1.00      1.00       200
           7       1.00      1.00      1.00       382
           8       1.00      1.00      1.00       190
           9       1.00      1.00      1.00       354
          10       1.00      1.00      1.00       335
          11       1.00      1.00      1.00       281
          12       1.00      1.00      1.00       642
          13       1.00      1.00      1.00       318
          14       1.00      1.00      1.00        75

    accuracy                           1.00      4128
   macro avg       1.00      1.00      1.00      4128
weighted avg

In [16]:
def predict_disease(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([predicted_class])[0]

In [17]:
predict_disease("Tomato leaf showing yellow spots and fungal infection")

'Tomato Late blight'

###Save model

In [18]:
# Save model
model.save_pretrained("./plant_disease_text_model")

# Save tokenizer
tokenizer.save_pretrained("./plant_disease_text_model")

# Save label encoder
import pickle

with open("./plant_disease_text_model/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

model = DistilBertForSequenceClassification.from_pretrained("./plant_disease_text_model")
tokenizer = DistilBertTokenizerFast.from_pretrained("./plant_disease_text_model")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]